<a href="https://colab.research.google.com/github/shah833/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shah833/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

if not os.path.isdir("Flyrank-ML-Internship"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shah833/Flyrank-ML-Internship"])

os.chdir("Flyrank-ML-Internship")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.run([sys.executable, "scripts/run_all.py"])

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_all.py'], returncode=0)

In [2]:
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head(3)

30000 rows, 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


In [3]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Eligibility:** A page is eligible if it has at least 80 impressions and was last updated more than 2 months ago.

**Rule:** Among eligible pages, if a page's CTR is at least 85% lower than similar-position peers, flag it for review.

**Reason code:** CTR_BELOW_POSITION_PEERS

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
eligible = df[(df["impressions_90d"] >= 80) & (df["days_since_last_update"] > 60)].copy()

In [5]:
eligible["peer_ctr"] = eligible.groupby("position_tier")["ctr"].transform("mean")

In [6]:
eligible["ctr_gap_pct"] = (eligible["peer_ctr"] - eligible["ctr"]) / eligible["peer_ctr"]

In [7]:
eligible["flagged"] = eligible["ctr_gap_pct"] >= 0.85

In [8]:
eligible[["content_id", "position_tier", "ctr", "peer_ctr", "ctr_gap_pct", "flagged"]].head(10)

,content_id,position_tier,ctr,peer_ctr,ctr_gap_pct,flagged
9,content_c27558df2b0c,page_1,0.16,0.303048,0.472031,False
10,content_d8ee6cc6d642,top_3,1.55,0.317093,-3.888155,False
12,content_42fb2cad9ecf,page_1,1.76,0.303048,-4.807660,False
13,content_a5a2fbc76336,page_3_5,0.00,0.139486,1.000000,True
14,content_91067a14431a,page_3_5,0.00,0.139486,1.000000,True
16,content_78bd1d4a1d4d,page_1,0.15,0.303048,0.505029,False
24,content_0e23e310d404,page_1,0.27,0.303048,0.109052,False
27,content_7ea135180dd9,page_1,0.17,0.303048,0.439033,False
35,content_1a28b25c7128,page_3_5,0.03,0.139486,0.784925,False
38,content_dbe82879a406,page_3_5,0.12,0.139486,0.139701,False


In [9]:
print(eligible["flagged"].sum(), "pages flagged out of", len(eligible), "eligible pages")

2453 pages flagged out of 8252 eligible pages


In [10]:
eligible["score"] = eligible["ctr_gap_pct"]

In [11]:
eligible[["content_id", "ctr_gap_pct", "score"]].head(5)

,content_id,ctr_gap_pct,score
9,content_c27558df2b0c,0.472031,0.472031
10,content_d8ee6cc6d642,-3.888155,-3.888155
12,content_42fb2cad9ecf,-4.807660,-4.807660
13,content_a5a2fbc76336,1.000000,1.000000
14,content_91067a14431a,1.000000,1.000000


In [12]:
eligible["action"] = eligible["flagged"].map({True: "snippet_review", False: "monitor"})

In [13]:
eligible[["content_id", "flagged", "action"]].head(5)

,content_id,flagged,action
9,content_c27558df2b0c,False,monitor
10,content_d8ee6cc6d642,False,monitor
12,content_42fb2cad9ecf,False,monitor
13,content_a5a2fbc76336,True,snippet_review
14,content_91067a14431a,True,snippet_review


In [14]:
eligible["reason_code"] = eligible["flagged"].map({True: "CTR_BELOW_POSITION_PEERS", False: ""})

In [15]:
eligible[["content_id", "flagged", "reason_code"]].head(5)

,content_id,flagged,reason_code
9,content_c27558df2b0c,False,
10,content_d8ee6cc6d642,False,
12,content_42fb2cad9ecf,False,
13,content_a5a2fbc76336,True,CTR_BELOW_POSITION_PEERS
14,content_91067a14431a,True,CTR_BELOW_POSITION_PEERS


In [16]:
import os
os.makedirs("work/outputs", exist_ok=True)

In [17]:
queue = eligible[eligible["flagged"]].sort_values(["score", "impressions_90d"], ascending=[False, False])
queue = queue[["content_id", "position_tier", "ctr", "peer_ctr", "ctr_gap_pct", "score", "impressions_90d", "reason_code", "action"]]

queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Saved {len(queue)} flagged pages to baseline_action_score.csv")
queue.head(10)

Saved 2453 flagged pages to baseline_action_score.csv


,content_id,position_tier,ctr,peer_ctr,ctr_gap_pct,score,impressions_90d,reason_code,action
7445,content_c8e9d6ab9013,page_1,0.0,0.303048,1.0,1.0,208678,CTR_BELOW_POSITION_PEERS,snippet_review
8710,content_fb4bf6555c79,page_3_5,0.0,0.139486,1.0,1.0,84093,CTR_BELOW_POSITION_PEERS,snippet_review
26994,content_6e28a04c07a8,page_3_5,0.0,0.139486,1.0,1.0,41226,CTR_BELOW_POSITION_PEERS,snippet_review
25456,content_bc18d49d8f6b,page_3_5,0.0,0.139486,1.0,1.0,32491,CTR_BELOW_POSITION_PEERS,snippet_review
721,content_b21385c39124,page_3_5,0.0,0.139486,1.0,1.0,30962,CTR_BELOW_POSITION_PEERS,snippet_review
10136,content_df71843dcd17,deep,0.0,0.039842,1.0,1.0,27334,CTR_BELOW_POSITION_PEERS,snippet_review
11521,content_75175d878762,page_3_5,0.0,0.139486,1.0,1.0,25748,CTR_BELOW_POSITION_PEERS,snippet_review
18509,content_095661034f9b,page_3_5,0.0,0.139486,1.0,1.0,23513,CTR_BELOW_POSITION_PEERS,snippet_review
25462,content_825a9788af8d,page_1,0.0,0.303048,1.0,1.0,16786,CTR_BELOW_POSITION_PEERS,snippet_review
9443,content_8ba781dafa55,page_1,0.0,0.303048,1.0,1.0,16156,CTR_BELOW_POSITION_PEERS,snippet_review


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Action and Reason code:** Action and reason code is already meentioned in the table.

**Confidence tone:** Moderate — I trust this more because the page has a lot of impressions, not just a low CTR. But it's still based on only one signal (CTR vs. peers), not several signals agreeing together.

**What would make it wrong:** Maybe the clicks are happening but not being tracked correctly, a broken tracking script, not a real problem with the page.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)
top20

,content_id,position_tier,ctr,peer_ctr,ctr_gap_pct,score,impressions_90d,reason_code,action
7445,content_c8e9d6ab9013,page_1,0.0,0.303048,1.0,1.0,208678,CTR_BELOW_POSITION_PEERS,snippet_review
8710,content_fb4bf6555c79,page_3_5,0.0,0.139486,1.0,1.0,84093,CTR_BELOW_POSITION_PEERS,snippet_review
26994,content_6e28a04c07a8,page_3_5,0.0,0.139486,1.0,1.0,41226,CTR_BELOW_POSITION_PEERS,snippet_review
25456,content_bc18d49d8f6b,page_3_5,0.0,0.139486,1.0,1.0,32491,CTR_BELOW_POSITION_PEERS,snippet_review
721,content_b21385c39124,page_3_5,0.0,0.139486,1.0,1.0,30962,CTR_BELOW_POSITION_PEERS,snippet_review
10136,content_df71843dcd17,deep,0.0,0.039842,1.0,1.0,27334,CTR_BELOW_POSITION_PEERS,snippet_review
11521,content_75175d878762,page_3_5,0.0,0.139486,1.0,1.0,25748,CTR_BELOW_POSITION_PEERS,snippet_review
18509,content_095661034f9b,page_3_5,0.0,0.139486,1.0,1.0,23513,CTR_BELOW_POSITION_PEERS,snippet_review
25462,content_825a9788af8d,page_1,0.0,0.303048,1.0,1.0,16786,CTR_BELOW_POSITION_PEERS,snippet_review
9443,content_8ba781dafa55,page_1,0.0,0.303048,1.0,1.0,16156,CTR_BELOW_POSITION_PEERS,snippet_review


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** At first, all my zero-CTR pages had the same score, so they were in random order. I fixed this by using impressions as a tiebreaker, more impressions means stronger evidence. But it's not perfect: I still can't tell 'definitely broken' from 'probably a problem' within that group.

**Leakage check:** My rule only uses ctr, peer_ctr, impressions_90d, and days_since_last_update — none of which come from trend_direction or trend_pct, the columns I'm supposed to avoid since they're basically the answer. Everything I used is something I could know right now, not a peek at the future. No leakage here

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.